# 1D kCSD (adapted from `CSD to share to cursor.ipynb`)

Same kCSD maths as your working notebook:
- per unique `x` column → `KCSD1D`
- `y` normalized to \[0, 1\]
- temporal Gaussian smooth (`sigma=1`)
- `h=0.1`, `n_src=2*n_ch`
- interpolate CSD back onto electrode depths

**Kernel:** `Python (kCSD_env)` — check `sys.executable` points to `...\envs\kCSD_env\python.exe`.

For delta peak/trough data from MATLAB Estimate, only set `DELTA_ROOT` below.


In [ ]:
import sys
print("executable:", sys.executable)
if "kCSD_env" not in sys.executable.replace("\\", "/"):
    print("WARNING: this kernel does not look like kCSD_env — change kernel before continuing.")


In [ ]:
# Only if imports fail in kCSD_env (run once in a terminal, not every session):
#   conda activate kCSD_env
#   python -m pip install -U -r requirements.txt


In [ ]:

# ---------------------------------------------------------
# 0) IMPORTS  (same packages as the share notebook + pipeline helper)
# ---------------------------------------------------------
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from csd_1d_pipeline import (
    KCSD1D,
    DELTA_CSD_DEFAULTS,
    SHARE_NOTEBOOK_KCSD,
    discover_delta_csd_datasets,
    run_delta_csd_from_root,
    run_dataset,
)

# Fail early with a clear message if kcsd is missing in this env
assert KCSD1D is not None, (
    "KCSD1D could not be imported. Activate kCSD_env / select that kernel, "
    "then: python -m pip install -U -r requirements.txt"
)
print("KCSD1D OK:", KCSD1D)
print("Share-notebook kCSD settings:", SHARE_NOTEBOOK_KCSD)


## A) Delta peak/trough root (MATLAB Estimate output)

Only change this path. Uses the share-notebook kCSD settings with a
**−500 … +500 ms** window and **baseline correction** (pre-T0 mean).
No re-reference.


In [ ]:

# ---------------------------------------------------------
# 1) USER SETTINGS — delta export folder
# ---------------------------------------------------------
DELTA_ROOT = r"\\experimentfs.home.bccn-berlin.de\experiment\PlayNeuralData\NPX-OPTO PLAY NMM\PlayBout Analysis\play bout analysis behavior and physiology\Data\Analysis results\delta peak trough CSD"

datasets = discover_delta_csd_datasets(DELTA_ROOT)
print(f"Found {len(datasets)} peak/trough folders")
for d in datasets[:12]:
    print(" ", d["name"])
if len(datasets) > 12:
    print(" ...")


In [ ]:
# Window: -500 .. +500 ms around T0; baseline_correct=True (DELTA_CSD_DEFAULTS)

# ---------------------------------------------------------
# 2–5) LOAD → EPOCH → 1D kCSD per column → PLOT/SAVE
#      (same algorithm as cells 1–3 of the share notebook)
# ---------------------------------------------------------
results = run_delta_csd_from_root(
    DELTA_ROOT,
    plot=True,          # show example column like the share notebook
    stop_on_error=False,
)

ok = [r for r in results if "error" not in r]
bad = [r for r in results if "error" in r]
print(f"Finished: {len(ok)} ok, {len(bad)} failed")
for r in bad:
    print(" -", r["name"], ":", r["error"])


## B) Single Lombard/stim folder (exact share-notebook style)

Use this to re-run one of your old folders the same way as
`CSD to share to cursor.ipynb` (with reference + baseline).


In [ ]:

# ---------------------------------------------------------
# Optional: one stim folder, mimicking the share notebook settings
# ---------------------------------------------------------
# main_folder = r"//experimentfs.home.bccn-berlin.de/experiment/share/Miguel/LOMBARD EFFECT NPX files and analysis/CSD analysis/230516/USV CSD/"
# cfg = {
#     **SHARE_NOTEBOOK_KCSD,
#     "folder": main_folder,
#     "name": "230516_USV",
#     "t_pre": 200,
#     "t_post": 800,
#     "ref_chan": 0,
#     "channels_remove": (191,),
#     "chan_start": 0,
#     "chan_end": 383,
#     "baseline_correct": True,
# }
# single = run_dataset(cfg)
